# Notes
Closed-source frontier models:<br>
OpenAI - ChatGPT<br>
Anthropic - Claude<br>
Google - Gemini<br>
x.ai - Grok<br>
<br>
Open-source:<br>
Meta - Llama<br>
Mistral - Mixtral<br>
Alibaba Cloud - Qwen<br>
Google - Gemma<br>
Microsoft - Phi<br>
DeekSeek AI - DeekSeek<br>
OpenAi - GPT-OSS<br>
<br>
Three ways to use models:<br>
- chat interfaces<br>
- call LLMs running on the cloud using APIs<br>
- Direct running - locally<br>

#### Chat Completion
- simplest way to call an LLM
- works like predictive writing

In [2]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

**endpoint** - an http url you can call to make an api request by hitting a web address -> web address is endpoint

In [10]:
import requests

headers = {"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"}

payload = { #json chunk
    "model": "gpt-5-nano",
    "messages": [
        {"role": "user", "content": "Tell me a fun fact"}]
}

payload

{'model': 'gpt-5-nano',
 'messages': [{'role': 'user', 'content': 'Tell me a fun fact'}]}

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions", #endpoint
    headers=headers,
    json=payload
)

response.json() #basically just results in a lot of json

{'id': 'chatcmpl-DmuFMsuZ6pFJvoWnMBQDbm372W9l0',
 'object': 'chat.completion',
 'created': 1780548328,
 'model': 'gpt-5-nano-2025-08-07',
 'choices': [{'index': 0,
   'message': {'role': 'assistant',
    'content': 'Fun fact: On Venus, a day is longer than a year. It takes about 243 Earth days to rotate once, but only about 225 Earth days to complete one orbit around the Sun. So a Venusian day is longer than a Venusian year. Want another fun fact on a specific topic?',
    'refusal': None,
    'annotations': []},
   'finish_reason': 'stop'}],
 'usage': {'prompt_tokens': 11,
  'completion_tokens': 455,
  'total_tokens': 466,
  'prompt_tokens_details': {'cached_tokens': 0, 'audio_tokens': 0},
  'completion_tokens_details': {'reasoning_tokens': 384,
   'audio_tokens': 0,
   'accepted_prediction_tokens': 0,
   'rejected_prediction_tokens': 0}},
 'service_tier': 'default',
 'system_fingerprint': None}

In [ ]:
response.json()["choices"][0]["message"]["content"] #accessing the actual response content

'Fun fact: Bananas are berries, but strawberries aren’t. Botanically, a berry is a fruit from a single ovary with seeds baked inside the flesh. Bananas fit that definition, while strawberries are actually aggregate fruits made from multiple ovaries. Want another fun fact in a different category?'

#### openai package
- python client library
- wrapper around http endpoint call
- work with clean python code instead of json objects

In [14]:
#below does the same thing as the code above but is cleaner
#endpoint call over http

from openai import OpenAI
openai = OpenAI()

response = openai.chat.completions.create(model="gpt-5-nano", messages=[{"role": "user", "content": "Tell me a fun fact"}])

response.choices[0].message.content

'Fun fact: A day on Venus is longer than a year on Venus. It takes about 243 Earth days to rotate once, but only about 225 Earth days to orbit the Sun. Plus, Venus rotates in the opposite direction, so the Sun rises in the west. Want another one?'

### Ollama's OpenAI compatible endpoint

In [ ]:
!ollama pull llama3.2:1b #download the model

In [18]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

#get a fun fact
response = ollama.chat.completions.create(model="llama3.2:1b", messages=[{"role": "user", "content": "Tell me a fun fact"}])
response.choices[0].message.content

'Here\'s a fun fact: There is a type of jellyfish that is immortal. The Turritopsis dohrnii, also known as the "immortal jellyfish," can transform its body into a younger state through a process called transdifferentiation, essentially making it immune to aging and allowing it to regenerate its entire body from a polyp stage back to an adult form.'

**benefits of using an open source model**
- no api charges
- data doesn't leave your box<br>
<br>
**disadvantages**
- slightly less power than frontier model

# "Homework" - Website Summary (using Ollama!)
Basically the same thing as Project 1, but modifying it so that we use an open-source model (Ollama) rather than OpenAI

In [3]:
from openai import OpenAI
from scraper import fetch_website_contents

OLLAMA_BASE_URL = "http://localhost:11434/v1"
MODEL = "llama3.2:1b"

#same system and user prompts as before
system_prompt = """
You are a snarky assistant that analyzes the contents of a website,
and provides a short, snarky, humorous summary, ignoring text that might be navigation related.
Respond in markdown. Do not wrap the markdown in a code block - respond just with the markdown.
"""
user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

In [7]:
def messages_for(website):
    #Create message list for the LLM.
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]


def summarize(url):
    #Fetch and summarize a website using Ollama.
    ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create(
        model=MODEL,
        messages=messages_for(website)
    )
    return response.choices[0].message.content

def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

url = "https://edwarddonner.com"
summary = summarize(url)
print(summary)

*The AI startup world gets a bit dull. Here's a quick rundown:*

*   This is the founder's personal website, and they're all about exploring LLM technology with their "arena of diplomacy and deviousness" - aka an online game between LLMs.
*   You might find some news or announcements here, but it's been ages since anyone posted. Probably just the usual startup updates...
